In [1]:
import pandas as pd

rawsearch_engine_map = pd.read_csv("../resources/wum_dataset_hw/search_engine_map.csv")
rawvisitors = pd.read_csv("../resources/wum_dataset_hw/visitors.csv")
rawclicks = pd.read_csv("../resources/wum_dataset_hw/clicks.csv")

In [2]:
# rawclicks[rawclicks["PageName"]=="HOWTOJOIN"]
# rawvisitors
# rawvisitors = rawvisitors[(rawvisitors["Length_pagecount"] >= 2)]
# rawvisitors
# rawclicks[rawclicks["VisitID"] ==  1226]
# rawclicks[(rawclicks["PageName"] == "lastminute")]


# rawclicks[(rawclicks["TopicName"] == "Lastminute")]
# rawvisitors[rawvisitors["VisitID"] ==  4643]
# rawvisitors[(rawvisitors["Length_seconds"] == 0) & (rawvisitors["Length_pagecount"] > 1)]

rawvisitors[(rawvisitors["Length_pagecount"] > 1)]

,VisitID,Referrer,Day,Hour,Length_seconds,Length_pagecount
0,1185,URI_9,Sunday,23,0,6
1,1186,URI_38,Sunday,23,300,2
2,1187,URI_0,Sunday,23,240,12
4,1189,URI_0,Monday,0,540,7
5,1190,URI_45,Monday,0,180,2
...,...,...,...,...,...,...
15550,17257,URI_0,Tuesday,14,300,3
15551,17258,URI_137,Tuesday,14,0,2
15553,17260,URI_0,Tuesday,14,60,4
15557,17264,URI_45,Tuesday,14,240,5


In [3]:
rawvisitors

,VisitID,Referrer,Day,Hour,Length_seconds,Length_pagecount
0,1185,URI_9,Sunday,23,0,6
1,1186,URI_38,Sunday,23,300,2
2,1187,URI_0,Sunday,23,240,12
3,1188,URI_45,Monday,0,0,1
4,1189,URI_0,Monday,0,540,7
...,...,...,...,...,...,...
15554,17261,URI_5,Tuesday,14,0,1
15555,17262,URI_111,Tuesday,14,0,1
15556,17263,URI_111,Tuesday,14,0,1
15557,17264,URI_45,Tuesday,14,240,5


## Step 1

In [4]:
import pandas as pd

clicks = pd.read_csv("../resources/wum_dataset_hw/clicks.csv")

# drop page ids, local ids, cat ids and so on
clicks = clicks[
    [
        "VisitID",
        "SequenceNumber",
        "PageName",
        "CatName",
        "ExtCatName",
        "TopicName",
        "TimeOnPage",
    ]
]

def normalize(x):
    if pd.isna(x):
        return ""
    y = str(x).strip().upper()
    y = y.replace(" ", "_")
    return y

for col in ["PageName", "CatName", "ExtCatName", "TopicName"]:
    clicks[col] = clicks[col].apply(normalize)

# sort to preserve navigation order sequence, strip string and replace spaces with _
clicks = clicks.sort_values(["VisitID", "SequenceNumber"])

visitors = pd.read_csv("../resources/wum_dataset_hw/visitors.csv")
visitors = visitors[(visitors["Length_pagecount"] >= 2) & (visitors["Length_seconds"] > 0)]
valid_ids = set(visitors["VisitID"])

clicks = clicks[clicks["VisitID"].isin(valid_ids)]


In [5]:
PAGE_TO_APRIORI_CONVERSION = {
    "APPLICATION": "VIEW_APPLICATION", # macro
    "CATALOG": "VIEW_CATALOG",

    "INSURANCE": "VIEW_INSURANCE", # micro
    "DISCOUNT": "VIEW_DISCOUNT",
    "WHOWEARE": "VIEW_WHOWEARE",
    "HOWTOJOIN": "VIEW_HOWTOJOIN",

    "LASTMINUTE": "VIEW_LASTMINUTE"
}

In [6]:
def process_session(session):
    items = set()

    topics = set()
    search_count = 0
    page_count = len(session)

    has_application = 0
    has_insurance = 0
    has_discount = 0
    has_catalog = 0
    has_whoweare = 0
    has_howtojoin = 0
    has_lastminute = 0
    for page, cat, ext, topic, time in session:

        # ---- TOPICS, remove those which don't make sense or provide "INFO"----
        # lastminute is added as VIEW_LASTMINUTE
        if topic not in {"IN_GENERAL", "UNDETECTED", "NENI", "LASTMINUTE", ""}:
            topics.add(topic)

        # ---- how many searches ----
        if cat == "SEARCH":
            search_count += 1

        # ---- CONVERSION / HELP PAGES ----
        mapped = None

        # using only pages as cat names don't include all (at least HOWTOJOIN)
        if page in PAGE_TO_APRIORI_CONVERSION:
            mapped = PAGE_TO_APRIORI_CONVERSION[page]

        if mapped:
            items.add(mapped)

            if mapped == "VIEW_APPLICATION":
                has_application = 1
            elif mapped == "VIEW_INSURANCE":
                has_insurance = 1
            elif mapped == "VIEW_DISCOUNT":
                has_discount = 1
            elif mapped == "VIEW_CATALOG":
                has_catalog = 1
            elif mapped == "VIEW_WHOWEARE":
                has_whoweare = 1
            elif mapped == "VIEW_HOWTOJOIN":
                has_howtojoin = 1
            elif mapped == "VIEW_LASTMINUTE":
                has_lastminute = 1

    # ---- ADD TOPICS TO TRANSACTION ----
    for t in topics:
        items.add(f"TOPIC_{t}")

    # ---- BEHAVIORAL FEATURES AS ITEMS ----
    for n in reversed(range(1, 9)):
        if len(topics) >= n:
            items.add(f"TOPICS>={n}")
            break
    for n in reversed(range(1, 9)):
        if search_count >= n:
            items.add(f"SEARCH>={n}")
            break

    # ---- FEATURE VECTOR ----
    features = {
        "Has_APPLICATION": has_application,
        "Has_INSURANCE": has_insurance,
        "Has_DISCOUNT": has_discount,
        "Has_CATALOG": has_catalog,
        "Has_WHOWEARE": has_whoweare,
        "Has_HOWTOJOIN": has_howtojoin,
        "Has_LASTMINUTE": has_lastminute,
        "SearchCount": search_count,
        "PageCount": page_count,
        "TopicCount": len(topics),
    }

    return items, features

In [7]:
def build_sessions(df):
    return list(zip(
        df["PageName"],
        df["CatName"],
        df["ExtCatName"],
        df["TopicName"],
        df["TimeOnPage"]
    ))

sessions = clicks.groupby("VisitID").apply(build_sessions, include_groups=False)
sessions.reset_index(name='actions')

,VisitID,actions
0,1186,"[(NORWAY_JOURNEY_TO_THE_FAR_NORTH_LOFOTEN, TRI..."
1,1187,"[(TRAVELAGENCY, INFO, HOMEPAGE, IN_GENERAL, 30..."
2,1189,"[(TRAVELAGENCY, INFO, HOMEPAGE, IN_GENERAL, 60..."
3,1190,"[(N_NEWS_TRIPS, INFO, SENDS_NEWS, IN_GENERAL, ..."
4,1192,"[(TRAVELAGENCY, INFO, HOMEPAGE, IN_GENERAL, 30..."
...,...,...
5537,17255,"[(TRAVELAGENCY, INFO, HOMEPAGE, IN_GENERAL, 30..."
5538,17257,"[(TRAVELAGENCY, INFO, HOMEPAGE, IN_GENERAL, 60..."
5539,17260,"[(TRAVELAGENCY, INFO, HOMEPAGE, IN_GENERAL, 30..."
5540,17264,"[(BULGARIA_STAY_WITH_HOLIDAY_TRIPS_05, TRIP, U..."


In [8]:
transaction_list = []
feature_rows = []

for visit_id, session in sessions.items():
    items, features = process_session(session)
    transaction_list.append(
        {"VisitId": visit_id, "items": items}
    )

    features["VisitID"] = visit_id
    feature_rows.append(features)

transactions_df = pd.DataFrame(transaction_list)
features_df = pd.DataFrame(feature_rows)

In [9]:
(features_df["Has_CATALOG"] == True).value_counts()

Has_CATALOG
False    5087
True      455
Name: count, dtype: int64

In [10]:
(features_df["Has_APPLICATION"] == True).value_counts()

Has_APPLICATION
False    5526
True       16
Name: count, dtype: int64

In [36]:
transactions = [t["items"] for t in transaction_list]
transactions = [
    t for t in transactions
    if len(t) > 1
]
print("Transactions count:", len(transactions))

Transactions count: 5211


In [12]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

df_onehot = pd.DataFrame(te_array, columns=te.columns_)

In [31]:
from mlxtend.frequent_patterns import apriori

frequent_itemsets = apriori(
    df_onehot,
    min_support=0.001,
    use_colnames=True
)

In [39]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.20
)


In [40]:
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(TOPICS>=1),(SEARCH>=1),0.458837,0.370754,0.272309,0.593476,1.600725,1.0,0.102193,1.547867,0.693475,0.488636,0.353950,0.663974
1,(SEARCH>=1),(TOPICS>=1),0.370754,0.458837,0.272309,0.734472,1.600725,1.0,0.102193,2.038064,0.596401,0.488636,0.509338,0.663974
2,(TOPICS>=2),(SEARCH>=1),0.282096,0.370754,0.058530,0.207483,0.559624,1.0,-0.046058,0.793984,-0.522929,0.098482,-0.259471,0.182675
3,(TOPIC_BULGARIA),(SEARCH>=1),0.070620,0.370754,0.018998,0.269022,0.725607,1.0,-0.007184,0.860827,-0.289213,0.044980,-0.161674,0.160132
4,(TOPIC_CORSICA),(SEARCH>=1),0.104970,0.370754,0.025331,0.241316,0.650879,1.0,-0.013587,0.829391,-0.374723,0.056242,-0.205703,0.154820
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38315,"(TOPIC_TOURS_WITH_ACCOMMODATION, SEARCH>=8, TO...","(TOPIC_STAYS_WITH_TRIPS, VIEW_LASTMINUTE, TOPI...",0.004798,0.002303,0.001151,0.240000,104.220000,1.0,0.001140,1.312759,0.995179,0.193548,0.238246,0.370000
38316,"(TOPIC_HOTELBUSES, SEARCH>=8, TOPIC_TOURS_WITH...","(TOPIC_STAYS_WITH_TRIPS, VIEW_LASTMINUTE, TOPI...",0.004414,0.002303,0.001151,0.260870,113.282609,1.0,0.001141,1.349826,0.995567,0.206897,0.259164,0.380435
38317,"(VIEW_LASTMINUTE, TOPIC_HOTELBUSES, TOPIC_TOUR...","(TOPIC_STAYS_WITH_TRIPS, TOPIC_TOURISM, SEARCH...",0.002111,0.004414,0.001151,0.545455,123.581028,1.0,0.001142,2.190290,0.994006,0.214286,0.543439,0.403162
38318,"(VIEW_LASTMINUTE, TOPIC_HOTELBUSES, TOPIC_TOUR...","(TOPIC_STAYS_WITH_TRIPS, TOPIC_TOURISM, SEARCH...",0.005565,0.002879,0.001151,0.206897,71.875862,1.0,0.001135,1.257240,0.991606,0.157895,0.204607,0.303448


In [41]:
rules_catalog = rules[
    rules["consequents"].apply(lambda x: "VIEW_CATALOG" in x)
]
rules_catalog.sort_values(by=["lift"], ascending=False)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
3795,"(VIEW_WHOWEARE, TOPIC_ALPS)",(VIEW_CATALOG),0.002111,0.073115,0.001343,0.636364,8.703651,1.0,0.001189,2.548935,0.886978,0.018182,0.607679,0.327368
15763,"(TOPIC_TOURISM, VIEW_DISCOUNT, TOPIC_TOURS_WIT...",(VIEW_CATALOG),0.002495,0.073115,0.001151,0.461538,6.312538,1.0,0.000969,1.721359,0.843690,0.015464,0.419064,0.238643
5057,"(VIEW_WHOWEARE, TOPIC_TOURS_WITH_ACCOMMODATION)",(VIEW_CATALOG),0.003838,0.073115,0.001727,0.450000,6.154724,1.0,0.001446,1.685246,0.840750,0.022959,0.406615,0.236811
15784,"(VIEW_LASTMINUTE, VIEW_DISCOUNT, TOPIC_TOURS_W...",(VIEW_CATALOG),0.002687,0.073115,0.001151,0.428571,5.861642,1.0,0.000955,1.622050,0.831634,0.015424,0.383496,0.222160
4660,"(TOPIC_HIKING, VIEW_WHOWEARE)",(VIEW_CATALOG),0.004030,0.073115,0.001343,0.333333,4.559055,1.0,0.001049,1.390328,0.783815,0.017722,0.280745,0.175853
2921,"(TOPICS>=6, TOPIC_CYCLO)",(VIEW_CATALOG),0.003454,0.073115,0.001151,0.333333,4.559055,1.0,0.000899,1.390328,0.783362,0.015267,0.280745,0.174541
5018,"(VIEW_WHOWEARE, TOPIC_TOURISM)",(VIEW_CATALOG),0.006908,0.073115,0.002303,0.333333,4.559055,1.0,0.001798,1.390328,0.786087,0.029630,0.280745,0.182415
5015,"(TOPIC_TOURISM, VIEW_DISCOUNT)",(VIEW_CATALOG),0.008252,0.073115,0.002687,0.325581,4.453031,1.0,0.002083,1.374347,0.781886,0.034146,0.272382,0.181163
15615,"(VIEW_LASTMINUTE, TOPIC_LIPARI, TOPIC_TOURS_WI...",(VIEW_CATALOG),0.004798,0.073115,0.001535,0.320000,4.376693,1.0,0.001184,1.363067,0.775236,0.020101,0.266360,0.170499
5209,"(TOPICS>=1, SEARCH>=2, VIEW_WHOWEARE)",(VIEW_CATALOG),0.003646,0.073115,0.001151,0.315789,4.319105,1.0,0.000885,1.354679,0.771283,0.015228,0.261818,0.165769


In [42]:
0.005  * len(transactions)

26.055

In [43]:
rules_app = rules[
    rules["consequents"].apply(lambda x: "VIEW_APPLICATION" in x)
]
rules_app

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
